# VOSIM и FOF — форманта без фильтра

Два старых способа получить «голос» не фильтром, а **формой каждого периода**. Оба — родня пульсарного синтеза (Firmamentum): на каждый период рождается короткое событие, его внутренняя частота — форманта, частота повторения — высота.

| | VOSIM (Kaegi & Tempelaars, 1978) | FOF (Rodet, IRCAM CHANT, 1984) |
|---|---|---|
| событие | пачка из N импульсов sin², каждый тише в b раз, потом тишина | затухающий синус с мягкой атакой |
| форманта | 1 / длина импульса T | частота синуса fc |
| ширина форманты | число импульсов N и затухание b | скорость затухания (bw, Гц) |
| характер | сухой, «электронный голос», вау, гнусавость | живое пение, колокола, при низкой высоте — гранулы |
| цена | копейки: одна таблица sin² | по зерну на форманту на период, зёрна перекрываются |

Ячейки можно просто читать сверху вниз: звук уже вшит в выходы. Чтобы покрутить самому — запустить ядро (Python 3 + numpy, scipy, matplotlib; в конце — ползунки, нужны `ipywidgets`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from IPython.display import Audio, display

SR = 22050          # хватает: форманты ниже 5 кГц, а ноутбук легче
plt.rcParams.update({'figure.figsize': (11, 3), 'axes.grid': True, 'grid.alpha': .25})

def as_f(v):
    '''число или функция времени t (с) -> функция'''
    return v if callable(v) else (lambda t: v)

def dc_block(x, r=0.995):
    return lfilter([1, -1], [1, -r], x)

def fade(x, ms=15, sr=SR):
    n = int(ms * sr / 1000); w = np.ones(len(x))
    w[:n] = np.linspace(0, 1, n); w[-n:] = np.linspace(1, 0, n)
    return x * w

def play(x, sr=SR):
    x = fade(dc_block(x), sr=sr)
    display(Audio(x / (np.abs(x).max() + 1e-9) * 0.9, rate=sr, normalize=False))

def spec_db(x, sr=SR):
    w = np.hanning(len(x)); X = np.abs(np.fft.rfft(x * w))
    return np.fft.rfftfreq(len(x), 1 / sr), 20 * np.log10(X / X.max() + 1e-9)

def show(x, title='', ms=30, fmax=5000, sr=SR, start=0.3):
    '''слева — осциллограмма (первые ms), справа — спектр куска'''
    x = dc_block(x)
    i0 = int(start * sr) if len(x) > start * sr * 2 else 0
    fig, (a, b) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [1, 1.3]})
    seg = x[i0:i0 + int(ms * sr / 1000)]
    a.plot(np.arange(len(seg)) / sr * 1000, seg, lw=.8, color='k'); a.set_xlabel('мс'); a.set_title(title)
    f, d = spec_db(x[i0:i0 + 8192] if len(x) - i0 > 8192 else x, sr)
    b.plot(f, d, lw=.7, color='k'); b.set_xlim(0, fmax); b.set_ylim(-80, 3); b.set_xlabel('Гц'); b.set_ylabel('дБ')
    plt.tight_layout(); plt.show()

def sgram(x, title='', fmax=5000, sr=SR):
    plt.figure(figsize=(11, 2.8))
    plt.specgram(dc_block(x), NFFT=1024, Fs=sr, noverlap=896, cmap='Greys', vmin=-110)
    plt.ylim(0, fmax); plt.title(title); plt.xlabel('с'); plt.ylabel('Гц'); plt.grid(False); plt.show()

## Часть 1. VOSIM

**Один период:** N импульсов формы sin², каждый длиной T, амплитуда каждого следующего умножается на b; потом пауза M до конца периода.

- высота: f0 = 1 / (N·T + M) — здесь задаём f0 напрямую, пауза получается сама;
- форманта: около 1/T — у импульса sin² длины T основная энергия на частоте 1/T;
- N и b — насколько форманта «узкая»: много одинаковых импульсов = почти синус на 1/T, один импульс = широкий горб.

Все параметры могут быть числом или функцией времени — читаются один раз за период (как в оригинальном железе: значения меняются только на границе периода, поэтому без щелчков).

In [ ]:
def vosim(dur, f0, T, N=3, b=0.8, jitter=0.0, seed=0, sr=SR):
    '''f0 — высота, Гц; T — длина импульса, с (форманта ~ 1/T); N — импульсов; b — затухание;
    jitter — случайный разброс длины периода (0..1) — у Кэги так делались шумы'''
    rng = np.random.default_rng(seed)
    f0, T, N, b, jitter = map(as_f, (f0, T, N, b, jitter))
    n = int(dur * sr); out = np.zeros(n + sr)
    pos = 0.0
    while pos < n:
        t = pos / sr
        Tp, k_n, bb = T(t), max(1, int(round(N(t)))), b(t)
        i = int(pos) + 1
        L = int(np.ceil(k_n * Tp * sr))
        tt = (np.arange(L) + (i - pos)) / sr          # точное время от начала пачки
        k = np.floor(tt / Tp); ph = tt / Tp - k
        out[i:i + L] += (bb ** k) * np.sin(np.pi * ph) ** 2 * (k < k_n)
        per = sr / f0(t) * (1 + jitter(t) * rng.uniform(-1, 1))
        pos += max(per, 1.0)
    return out[:n]

# анатомия одного периода
x = vosim(0.05, f0=100, T=1/800, N=4, b=0.7)
plt.plot(np.arange(len(x[:int(SR/100)+2])) / SR * 1000, x[:int(SR/100)+2], 'k', lw=1)
plt.title('один период VOSIM: 4 импульса sin², b = 0.7, дальше пауза'); plt.xlabel('мс'); plt.show()

### 1.1 Базовый звук: высота 110 Гц, форманта 800 Гц

In [ ]:
x = vosim(2.5, f0=110, T=1/800, N=3, b=0.8)
show(x, 'f0 = 110, 1/T = 800')
play(x)

На спектре — гармоники 110 Гц под горбом с вершиной около 800 Гц. Это и есть форманта, без единого фильтра.

### 1.2 Вау без фильтра: двигаем T, высота стоит
Форманта едет 250 → 3000 Гц и обратно. Звучит как wah или губы, говорящие «уа-уа».

In [ ]:
D = 6
fm = lambda t: 250 * 12 ** (0.5 - 0.5 * np.cos(2 * np.pi * t / D))   # 250..3000 Гц
x = vosim(D, f0=98, T=lambda t: 1 / fm(t), N=3, b=0.8)
sgram(x, 'форманта едет, высота 98 Гц стоит')
play(x)

### 1.3 Обратное: высота едет, форманта стоит
Как голос, который поёт глиссандо одной гласной: тембр «прибит», ноты скользят. Хорошо слышно, чем это отличается от простого пилообразного глиссандо через фильтр — тут нет «звона» фильтра.

In [ ]:
x = vosim(5, f0=lambda t: 55 * 2 ** (t / 5 * 3), T=1/700, N=3, b=0.8)   # 55 -> 440 Гц
sgram(x, 'f0: 55 -> 440, форманта 700')
play(x)

### 1.4 N и b — ширина и «гнусавость» форманты
Четыре варианта подряд, по секунде: N=1 (один импульс, широкий горб, ярко), N=3 b=0.8, N=6 b=0.95 (почти синус на форманте — гнусаво, как варган), N=6 b=0.5.

In [ ]:
vars_ = [(1, 1.0), (3, 0.8), (6, 0.95), (6, 0.5)]
fig, ax = plt.subplots(1, 4, figsize=(13, 2.6), sharey=True)
xs = []
for a, (N, b) in zip(ax, vars_):
    x = vosim(1.2, f0=110, T=1/900, N=N, b=b); xs.append(fade(x))
    f, d = spec_db(dc_block(x)[2000:2000 + 8192]); a.plot(f, d, 'k', lw=.6)
    a.set_xlim(0, 4000); a.set_ylim(-70, 3); a.set_title(f'N={N}, b={b}')
plt.tight_layout(); plt.show()
play(np.concatenate(xs))

### 1.5 Ломаем: пачка длиннее периода
Если N·T > 1/f0, следующая пачка начинается, пока не кончилась предыдущая — импульсы налезают друг на друга. В железе это была ошибка, а у нас — звук: высота растёт, и примерно на 270 Гц (N·T = 3/800 с) VOSIM переходит в жёсткий гудящий режим.

In [ ]:
x = vosim(6, f0=lambda t: 80 * 2 ** (t / 6 * 3), T=1/800, N=3, b=0.9)   # 80 -> 640
sgram(x, 'f0 80 -> 640; после ~270 Гц пачки налезают')
play(x)

### 1.6 Шум из VOSIM: случайная длина периода
Кэги так получал согласные: длина паузы случайна. Разброс 0 → 100 % за 6 секунд: тон → хриплый голос → шипение, окрашенное формантой.

In [ ]:
x = vosim(6, f0=110, T=1/1500, N=2, b=0.7, jitter=lambda t: (t / 6) ** 1.5)
sgram(x, 'jitter 0 -> 1')
play(x)

### 1.7 Гласные из двух VOSIM
Одна VOSIM — одна форманта. Две на одной высоте (F1 + F2) уже дают узнаваемые гласные: а — о — у — и — э.

In [ ]:
VOW2 = {'a': (700, 1100), 'o': (450, 800), 'u': (320, 700), 'i': (280, 2300), 'e': (500, 1800)}
seq = 'aouie'; d = 0.9; xs = []
for v in seq:
    f1, f2 = VOW2[v]
    xs.append(fade(vosim(d, 110, 1/f1, 3, .8) + 0.5 * vosim(d, 110, 1/f2, 2, .7), 30))
x = np.concatenate(xs)
sgram(x, ' — '.join(seq))
play(x)

### 1.8 Набросок дрона в духе touch2decay
Три голоса (квинта + октава), у каждого своя медленная «дышащая» форманта, чуть-чуть случайности в периоде. Сыро, но слышно, куда это может пойти.

In [ ]:
D = 10
voices = [(55, 0.07, 400, 1200), (82.5, 0.05, 600, 1800), (110, 0.11, 300, 2600)]
x = 0
for f, rate, lo, hi in voices:
    fm = lambda t, r=rate, lo=lo, hi=hi: lo * (hi / lo) ** (0.5 - 0.5 * np.cos(2 * np.pi * r * t))
    x = x + vosim(D, f, lambda t, fm=fm: 1 / fm(t), N=4, b=0.85, jitter=0.01, seed=int(f))
sgram(x, 'три VOSIM-голоса с дышащими формантами', fmax=3500)
play(x)

## Часть 2. FOF — Forme d'Onde Formantique

**Одно зерно:** синус на частоте форманты fc, затухает экспонентой e^(−π·bw·t), начало сглажено полукосинусом длиной tex.

- bw (Гц) — ширина форманты: быстро гаснет = широкая форманта, долго звенит = узкая;
- tex — атака зерна: короче = шире «юбка» форманты (больше шипящих верхов), длиннее = чище;
- на каждый период высоты рождается новое зерно на каждую форманту; зёрна перекрываются.

Важное свойство: зерно живёт своей жизнью. Если высоту опустить ниже ~30 Гц, зёрна перестают сливаться в тон и становятся отдельными событиями — голос превращается в ритм, капли, колокола. Это и есть мост к гранулярному синтезу.

In [ ]:
def fof_grain(fc, bw, tex, sr=SR, t0=0.0, max_dur=0.5):
    dur = min(6.9 / (np.pi * bw), max_dur)       # до -60 дБ
    t = np.arange(int(dur * sr) + 1) / sr + t0
    att = np.where(t < tex, 0.5 * (1 - np.cos(np.pi * np.clip(t / tex, 0, 1))), 1.0)
    return att * np.exp(-np.pi * bw * t) * np.sin(2 * np.pi * fc * t)

def fof(dur, f0, formants, tex=0.003, vib=0.0, vib_rate=5.5, jitter=0.0, seed=0, sr=SR):
    '''formants: [(fc, bw, ампл_дБ), ...] или функция t -> такой список.
    vib — глубина вибрато (доля), jitter — случайный разброс периода (доля)'''
    rng = np.random.default_rng(seed)
    f0, F, jitter = as_f(f0), as_f(formants), as_f(jitter)
    n = int(dur * sr); out = np.zeros(n + sr)
    pos = 0.0
    while pos < n:
        t = pos / sr; i = int(pos) + 1
        for fc, bw, adb in F(t):
            g = fof_grain(fc, bw, tex, sr, t0=(i - pos) / sr)
            m = min(len(g), len(out) - i); out[i:i + m] += 10 ** (adb / 20) * g[:m]
        f = f0(t) * (1 + vib * np.sin(2 * np.pi * vib_rate * t))
        pos += max(sr / f * (1 + jitter(t) * rng.standard_normal()), 1.0)
    return out[:n]

# анатомия зерна: bw и tex
t = np.arange(int(0.03 * SR)) / SR * 1000
fig, ax = plt.subplots(1, 3, figsize=(13, 2.6), sharey=True)
for a, (bw, tex, ttl) in zip(ax, [(80, .003, 'bw 80, tex 3 мс'), (20, .003, 'bw 20 — звенит дольше'), (80, .0003, 'tex 0.3 мс — резкая атака')]):
    g = fof_grain(600, bw, tex, max_dur=0.03); m = min(len(g), len(t)); a.plot(t[:m], g[:m], 'k', lw=.7); a.set_title(ttl); a.set_xlabel('мс')
plt.tight_layout(); plt.show()

### 2.1 Пять формант — гласная «а» басом
Таблица формант — классическая, из документации Csound (бас и сопрано, частота / ширина / уровень).

In [ ]:
BASS = {
 'a': [(600, 60, 0), (1040, 70, -7), (2250, 110, -9), (2450, 120, -9), (2750, 130, -20)],
 'e': [(400, 40, 0), (1620, 80, -12), (2400, 100, -9), (2800, 120, -12), (3100, 120, -18)],
 'i': [(250, 60, 0), (1750, 90, -30), (2600, 100, -16), (3050, 120, -22), (3340, 120, -28)],
 'o': [(400, 40, 0), (750, 80, -11), (2400, 100, -21), (2600, 120, -20), (2900, 120, -40)],
 'u': [(350, 40, 0), (600, 80, -20), (2400, 100, -32), (2675, 120, -28), (2950, 120, -36)],
}
SOPRANO_A = [(800, 80, 0), (1150, 90, -6), (2900, 120, -32), (3900, 130, -20), (4950, 140, -50)]

x = fof(3, 110, BASS['a'], vib=0.006)
show(x, 'FOF, бас, «а», 110 Гц')
play(x)

### 2.2 Все гласные подряд, затем сопрано

In [ ]:
xs = [fade(fof(1.0, 110, BASS[v], vib=0.006), 40) for v in 'aeiou']
xs.append(fade(fof(2.0, 523, SOPRANO_A, vib=0.012, vib_rate=5.8), 60))
x = np.concatenate(xs)
sgram(x, 'бас: a e i o u, потом сопрано «а» на до второй октавы')
play(x)

### 2.3 Плавный морф гласных
Таблицы просто интерполируются во времени: а → и → у → а. Это то, чего VOSIM с двумя формантами делает грубее.

In [ ]:
def morph(path, D):
    keys = [np.array(BASS[v], float) for v in path]
    def F(t):
        u = np.clip(t / D, 0, 1) * (len(keys) - 1); k = min(int(u), len(keys) - 2); a = u - k
        a = 0.5 - 0.5 * np.cos(np.pi * a)
        return [tuple(r) for r in (1 - a) * keys[k] + a * keys[k + 1]]
    return F
D = 6
x = fof(D, 98, morph('aiua', D), vib=0.007)
sgram(x, 'а -> и -> у -> а')
play(x)

### 2.4 Ширина форманты: стекло и шёпот
Одна гласная, три ширины: все bw × 0.25 (узкие, звенящие, почти стеклянные), × 1, × 4 (широкие, мягкие, «ватные»).

In [ ]:
xs = []
for k in (0.25, 1, 4):
    F = [(fc, bw * k, a) for fc, bw, a in BASS['o']]
    xs.append(fade(fof(1.5, 110, F), 40))
x = np.concatenate(xs)
sgram(x, 'bw × 0.25, × 1, × 4')
play(x)

### 2.5 Главный трюк FOF: голос рассыпается в ритм
Высота падает 110 → 2 Гц. Сначала поёт, ниже ~30 Гц — «трещит», потом отдельные капли-удары, каждый — та же гласная, только мгновенная.

In [ ]:
D = 8
x = fof(D, lambda t: 110 * (2 / 110) ** (t / D), BASS['a'])
sgram(x, 'f0: 110 -> 2 Гц')
play(x)

### 2.6 Колокола и капли
Низкая «высота» (несколько Гц) со случайным разбросом и узкие высокие форманты → вместо голоса — дождь из колокольчиков. Те же 15 строк кода.

In [ ]:
BELL = [(880, 3, 0), (2420, 6, -8), (3960, 9, -14), (1320, 4, -6)]
x = fof(8, 3.5, BELL, tex=0.0004, jitter=0.6, seed=3)
x2 = fof(8, 2.3, [(fc * 0.75, bw, a) for fc, bw, a in BELL], tex=0.0004, jitter=0.7, seed=7)
x = x + 0.8 * x2
sgram(x, 'f0 ~3 Гц, jitter, узкие форманты')
play(x)

### 2.7 Гранулярное облако: высота — шум
Разброс периода на 100 % при 40 Гц — зёрна сыплются нерегулярно. Тон исчезает, остаётся облако, окрашенное гласной.

In [ ]:
x = fof(5, 40, morph('aoa', 5), jitter=lambda t: min(1.0, t / 3), seed=1)
sgram(x, 'jitter 0 -> 1, гласная «а-о-а»')
play(x)

### 2.8 Маленький хор
Шесть голосов FOF, у каждого своя расстройка, вибрато и микро-случайность — уже узнаваемый «хор» без единого семпла. Для сравнения с Paradisus.

In [ ]:
D = 7; x = 0
rng = np.random.default_rng(0)
for k in range(6):
    f = 110 * (1 + rng.uniform(-0.006, 0.006)) * (2 if k >= 4 else 1)
    x = x + fof(D, f, morph('aoa', D), vib=rng.uniform(0.004, 0.01), vib_rate=rng.uniform(4.8, 6.2),
                jitter=0.004, seed=k)
sgram(x, 'шесть голосов FOF')
play(x)

## Часть 3. Что с этим можно делать — и сколько стоит

**VOSIM**
- ручки: высота, форманта (1/T), N, b, разброс периода; всё меняется только на границе периода → щелчков нет даже при резких жестах;
- сильные места: «говорящий» дрон, вау без фильтра, ломание при N·T > периода, шум-согласные из разброса;
- слабое: одна форманта на голос; для гласных — 2–3 VOSIM на одной высоте;
- цена в AudioWorklet: одна таблица sin² и счётчик — дешевле любого фильтра. На iPhone — десятки голосов.

**FOF**
- ручки: высота, набор формант (fc, bw, уровень), tex, вибрато, разброс;
- сильные места: самое живое пение из синтеза; плавный морф гласных; при низкой высоте — ритм, капли, колокола, гранулярное облако. Один движок покрывает «голос ↔ ритм ↔ облако»;
- слабое: при высокой высоте и узких формантах зёрен в полёте много: 5 формант × (длина зерна × f0). При 110 Гц и bw 60 — ~20 зёрен на форманту одновременно, ~100 синусов. Терпимо, но ограничивать число голосов;
- удешевление: зерно — табличное (синус × готовая огибающая), или заменить зерно затухающим резонатором, возбуждаемым импульсом раз в период — звучит почти так же, это и есть «FOF через фильтры».

**Различие на слух:** VOSIM — сухой, синтетический, «робот-монах»; FOF — влажный, человеческий, и при этом умеет распадаться на зёрна.

## Покрутить самому (нужно запущенное ядро и `ipywidgets`)

In [ ]:
try:
    from ipywidgets import interact, FloatLogSlider, IntSlider, FloatSlider, Dropdown
    @interact(f0=FloatLogSlider(value=110, base=2, min=0, max=10, description='f0'),
              formant=FloatLogSlider(value=800, base=2, min=7, max=12, description='1/T'),
              N=IntSlider(value=3, min=1, max=10), b=FloatSlider(value=.8, min=.2, max=1, step=.01),
              jitter=FloatSlider(value=0, min=0, max=1, step=.01))
    def vosim_play(f0, formant, N, b, jitter):
        play(vosim(2, f0, 1 / formant, N, b, jitter))

    @interact(f0=FloatLogSlider(value=110, base=2, min=0, max=10, description='f0'),
              vowel=Dropdown(options=list('aeiou')), bw_k=FloatLogSlider(value=1, base=2, min=-3, max=3, description='bw ×'),
              tex_ms=FloatLogSlider(value=3, base=2, min=-3, max=4, description='tex, мс'),
              jitter=FloatSlider(value=0, min=0, max=1, step=.01))
    def fof_play(f0, vowel, bw_k, tex_ms, jitter):
        F = [(fc, bw * bw_k, a) for fc, bw, a in BASS[vowel]]
        play(fof(2, f0, F, tex=tex_ms / 1000, jitter=jitter, vib=0.006))
except ImportError:
    print('pip install ipywidgets — тогда здесь будут ползунки')